# Phase 2 Ridge — Explainability (Phase 3 Stream -1)

Sanity 通過後,問下一個問題:**ridge 學到的權重長什麼樣?**

對 10 個隨機 target 各做 K=10 ridge 後,觀察:
- $w_i$ 分佈(每個 target 有 10 個 weight)
- $w_i$ 與距離 $d_i$ 的關係
- $w_i$ 與訓練段 Pearson $r_i$ 的關係
- 截距 $b$ 的分佈

這份分析的目的:
1. 確認權重不是極端值(若有 weight = 5 而其他 = 0,可能擬合不穩)
2. 判斷 ridge 的「機制」——是不是純粹比 mean 多了 per-target 偏差校正?

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

from smart_pole.data.loader import load_pole_hourly
from smart_pole.runner.experiment import _corr_table
from smart_pole.neighbors.selector import _haversine
from smart_pole.visualization.plots import setup_chinese_font

PROJECT_ROOT = Path('..').resolve()
setup_chinese_font(PROJECT_ROOT / 'NotoSansCJKtc-Regular.otf')

In [ ]:
dataset = load_pole_hourly(
    cache_path=PROJECT_ROOT / 'data' / 'pole_hourly.parquet',
    station_info_path=PROJECT_ROOT / 'data' / 'MOENV_iot_station.csv',
    start='2025-12-04', end='2026-02-05',
)
T, S = dataset.values.shape
sid_to_idx = {s: i for i, s in enumerate(dataset.station_ids)}
train_end = int(T * 0.8)
train_slice = slice(0, train_end)
K = 10
ALPHA = 1.0
print(f'T={T} S={S}, train_end={train_end}, K={K}')

In [ ]:
# 用 runner 同邏輯的 corr table
ids_per, _ = _corr_table(dataset.station_ids, dataset.values, train_slice, K)

# 從可解的站隨機挑 10 個
rng = np.random.default_rng(42)
solvable = [i for i, ids in enumerate(ids_per) if ids is not None and dataset.station_ids[i] in dataset.coords]
chosen = rng.choice(solvable, size=10, replace=False)
print(f'隨機選 10 個 target: {[dataset.station_ids[i] for i in chosen]}')

In [ ]:
rows = []
for i in chosen:
    target = dataset.station_ids[i]
    nb_ids = ids_per[i][:K]
    n_idx = [sid_to_idx[s] for s in nb_ids]

    # 取訓練資料
    y_tr = dataset.values[train_slice, i]
    X_tr = dataset.values[train_slice, :][:, n_idx]
    finite = np.isfinite(y_tr) & np.isfinite(X_tr).all(axis=1)
    if finite.sum() < K + 8:
        continue
    Xf, yf = X_tr[finite], y_tr[finite]
    rdg = Ridge(alpha=ALPHA, fit_intercept=True).fit(Xf, yf)
    weights = rdg.coef_
    intercept = float(rdg.intercept_)

    # 距離與 Pearson r
    lon_t, lat_t = dataset.coords[target]
    lons = np.array([dataset.coords[s][0] for s in nb_ids])
    lats = np.array([dataset.coords[s][1] for s in nb_ids])
    dists = _haversine(lon_t, lat_t, lons, lats)

    rs = []
    for k in range(K):
        x = X_tr[:, k]
        m = np.isfinite(y_tr) & np.isfinite(x)
        if m.sum() < 24:
            rs.append(np.nan); continue
        yc, xc = y_tr[m] - y_tr[m].mean(), x[m] - x[m].mean()
        denom = float(np.sqrt((yc*yc).sum() * (xc*xc).sum()))
        rs.append(float((yc*xc).sum() / denom) if denom > 0 else np.nan)
    rs = np.array(rs)

    for k in range(K):
        rows.append({
            'target': target, 'neighbor_rank': k,
            'weight': float(weights[k]),
            'distance_m': float(dists[k]),
            'pearson_r': float(rs[k]),
            'intercept': intercept,
        })
df = pd.DataFrame(rows)
df.head(20)

In [ ]:
# 每 target 的權重和(若 ≈ 1 表 ridge 在做加權平均;若 < 1 表 ridge 用 intercept 補)
weight_sum = df.groupby('target')['weight'].sum()
intercept_per = df.groupby('target')['intercept'].first()
print('每 target 的 weight 總和（≈1 = 加權平均；< 1 + 高 intercept = 偏差補正主導）:')
for t, s in weight_sum.items():
    b = intercept_per[t]
    print(f'  {t}:  Σw_i = {s:+.3f}    b = {b:+.3f}')
print(f'\nΣw_i 平均 = {weight_sum.mean():.3f} ± {weight_sum.std():.3f}')
print(f'intercept 平均 = {intercept_per.mean():.3f} ± {intercept_per.std():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
for t, g in df.groupby('target'):
    ax.scatter(g['distance_m']/1000, g['weight'], alpha=0.7, s=40, label=t[:6])
ax.axhline(0, color='k', lw=0.5)
ax.axhline(1/K, color='r', lw=0.5, linestyle='--', label=f'1/K = {1/K}')
ax.set_xlabel('鄰站距離 (km)'); ax.set_ylabel('ridge weight $w_i$')
ax.set_title('$w_i$ vs $d_i$  (顏色=target)')
ax.legend(loc='upper right', fontsize=7, ncol=2)

ax = axes[1]
for t, g in df.groupby('target'):
    ax.scatter(g['pearson_r'], g['weight'], alpha=0.7, s=40)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Pearson $r_i$ (train 段)'); ax.set_ylabel('ridge weight $w_i$')
ax.set_title('$w_i$ vs $r_i$')

ax = axes[2]
ax.hist(intercept_per.values, bins=10, edgecolor='k')
ax.axvline(intercept_per.mean(), color='r', linestyle='--', label=f'mean = {intercept_per.mean():.2f}')
ax.set_xlabel('intercept $b$'); ax.set_ylabel('target 數')
ax.set_title('K=10 ridge intercept 分佈')
ax.legend()

fig.tight_layout()
fig.savefig(PROJECT_ROOT / 'results' / 'ridge_explainability.png', dpi=120)
plt.show()

In [ ]:
# 量化 w 與 r 的相關性、w 與 d 的相關性
from scipy.stats import spearmanr
rho_wr, p_wr = spearmanr(df['weight'], df['pearson_r'], nan_policy='omit')
rho_wd, p_wd = spearmanr(df['weight'], df['distance_m'], nan_policy='omit')
print(f'Spearman ρ(w, r) = {rho_wr:+.3f}  (p={p_wr:.3g})')
print(f'Spearman ρ(w, d) = {rho_wd:+.3f}  (p={p_wd:.3g})')
print('\n→ 若 ρ(w, r) 顯著為正：ridge 確實偏好高相關鄰居（與 corr_weighted 一致方向）')
print('→ 若 ρ(w, d) 顯著為負：ridge 偏好近距離鄰居（與 IDW 一致方向）')

## 結論(寫進 `results/docs/PHASE_1_2_RESULTS.md` 附錄)

(下游程式碼會輸出實際數字,結論文字在 doc 那邊寫)